In [2]:
# 导入必要模块：getpass用于安全地输入API密钥，os模块用于设置环境变量
import getpass
import os

# 使用 getpass 安全地输入 OpenAI 的 API 密钥，并将其设置为环境变量
os.environ["OPENAI_API_KEY"] = getpass.getpass()
# 设置一个常见浏览器的 USER_AGENT
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
# 导入 LangChain 中的 ChatOpenAI 模块，用于调用 OpenAI 的模型
from langchain_openai import ChatOpenAI

# 使用 GPT-4o-mini 模型进行生成
llm = ChatOpenAI(model="gpt-4o-mini")

# 导入 bs4（Beautiful Soup 4）模块，用于解析 HTML 页面
import bs4

# 导入 LangChain 中的相关模块
from langchain import hub
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 加载指定网页的内容，并提取目标部分
# 这里使用 WebBaseLoader 加载 Lilian Weng 博客文章的内容
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

# 使用 loader 加载网页内容，得到包含文档内容的 docs 对象
docs = loader.load()

# 创建 RecursiveCharacterTextSplitter，用于将长文档切分为多个小的文本块
# chunk_size 表示每个块的最大字符数，chunk_overlap 表示块之间的重叠字符数
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# 使用 text_splitter 将文档切分成小块
splits = text_splitter.split_documents(docs)

# 创建向量存储，利用文档分块并通过 OpenAIEmbeddings 进行嵌入
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

# 创建检索器，用于在向量存储中检索与输入查询相关的文档片段
retriever = vectorstore.as_retriever()

# 从 LangChain Hub 中加载 RAG 模式下的提示模板
prompt = hub.pull("rlm/rag-prompt")


# 定义格式化文档的函数，用于将多个文档块拼接为一个字符串
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# 定义 RAG 管道，主要包括以下流程：
# 1. 使用 retriever 从向量存储中检索与输入问题相关的文档片段，并使用 format_docs 函数格式化这些文档片段
# 2. 将用户的问题传递到 RAG 提示模板中
# 3. 使用指定的 LLM（这里是 GPT-4o-mini）生成回答
# 4. 将生成的输出字符串化，以便最终输出
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 调用 rag_chain 并输入问题 "What is Task Decomposition?" 来获取回答
rag_chain.invoke("What is Task Decomposition?")#Decomposition中文含义是分解


C:\Users\41507\AppData\Roaming\Python\Python312\site-packages\langsmith\client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


'Task Decomposition refers to breaking down complex tasks into smaller, more manageable steps to facilitate easier processing and problem-solving. This can be achieved using techniques like Chain of Thought (CoT), which encourages models to "think step by step," or through methods like Tree of Thoughts that explore multiple reasoning possibilities. It can be implemented via simple prompts, task-specific instructions, or human input.'